# Rating Model Training

In [ ]:
"""
Train the player rating model.

This script trains a Random Forest regression pipeline for player match ratings,
evaluates it with leakage-aware splits, and stores the final model as
rating_model.pkl.
"""

from pathlib import Path
import sys
from typing import Iterable

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

sys.path.append(str(Path.cwd().parent))

from ml.toolkit.ml_utilities import (
    build_preprocessor,
    evaluate_model,
    prepare_features_and_target,
    print_evaluation,
    save_model,
    split_data_by_group,
    split_data_randomly,
)


DATA_PATH = Path("../data/transform/pro/stats_with_rating.csv")
MODEL_PATH = Path("rating_model.pkl")
TARGET_COLUMN = "rating"
PLAYER_GROUP_COLUMN = "player_id"
MATCH_GROUP_COLUMN = "match_id"

METADATA_COLUMNS = [
    "player_id",
    "match_id",
    "club_id",
    "player_name",
    "date",
    "home_club_id",
    "away_club_id",
    "home_goals",
    "away_goals",
]

NUMERIC_COLUMNS = [
    "goals",
    "assists",
    "minutes",
    "on_min",
    "off_min",
    "team_goals",
    "team_conceded",
]

CATEGORICAL_COLUMNS = [
    "position",
    "result",
]

BOOLEAN_COLUMNS = [
    "yellow",
    "yellow_red",
    "red",
    "start_eleven",
]

FEATURE_COLUMNS = NUMERIC_COLUMNS + BOOLEAN_COLUMNS + CATEGORICAL_COLUMNS

MODEL_PARAMETERS = {
    "n_estimators": 1000,
    "max_depth": 20,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "random_state": 42,
    "n_jobs": -1,
}


def validate_columns(dataframe: pd.DataFrame, required_columns: Iterable[str]) -> None:
    """Raise an error when required columns are missing from a dataframe."""
    missing_columns = sorted(set(required_columns) - set(dataframe.columns))

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")


def load_training_data(input_path: Path) -> pd.DataFrame:
    """Load the training dataset from disk."""
    if not input_path.exists():
        raise FileNotFoundError(f"Training data not found: {input_path}")

    return pd.read_csv(input_path)


def build_rating_model() -> Pipeline:
    """Build the rating prediction pipeline."""
    preprocessor = build_preprocessor(
        numeric_columns=NUMERIC_COLUMNS,
        categorical_columns=CATEGORICAL_COLUMNS,
        boolean_columns=BOOLEAN_COLUMNS,
    )

    return Pipeline(
        steps=[
            ("preprocessing", preprocessor),
            ("model", RandomForestRegressor(**MODEL_PARAMETERS)),
        ]
    )


def prepare_training_data(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Prepare rating features and target values."""
    required_columns = FEATURE_COLUMNS + METADATA_COLUMNS + [TARGET_COLUMN]
    validate_columns(dataframe, required_columns)

    features, target = prepare_features_and_target(
        dataframe=dataframe,
        target_column=TARGET_COLUMN,
        columns_to_drop=METADATA_COLUMNS,
    )

    return features[FEATURE_COLUMNS], target


def evaluate_split(
    model: Pipeline,
    features_train: pd.DataFrame,
    features_test: pd.DataFrame,
    target_train: pd.Series,
    target_test: pd.Series,
    split_name: str,
) -> dict[str, float]:
    """Train and evaluate a model on one split."""
    model.fit(features_train, target_train)
    predictions = model.predict(features_test)
    metrics = evaluate_model(target_test, predictions)
    print_evaluation(metrics, split_name)
    return metrics


def evaluate_model_splits(
    dataframe: pd.DataFrame,
    features: pd.DataFrame,
    target: pd.Series,
) -> None:
    """Evaluate the rating model with random, player, and match splits."""
    random_split = split_data_randomly(features, target)
    player_split = split_data_by_group(
        dataframe=dataframe,
        features=features,
        target=target,
        group_column=PLAYER_GROUP_COLUMN,
    )
    match_split = split_data_by_group(
        dataframe=dataframe,
        features=features,
        target=target,
        group_column=MATCH_GROUP_COLUMN,
    )

    evaluate_split(build_rating_model(), *random_split, split_name="Random Split")
    evaluate_split(build_rating_model(), *player_split, split_name="Player Split")
    match_metrics = evaluate_split(
        build_rating_model(),
        *match_split,
        split_name="Match Split",
    )

    print(f"\nR2 Match Split: {match_metrics['r2_score']:.4f}")


def train_final_model(features: pd.DataFrame, target: pd.Series) -> Pipeline:
    """Train the final rating model on all available training data."""
    model = build_rating_model()
    model.fit(features, target)
    return model


def main() -> None:
    """Run model training, evaluation, and model export."""
    dataframe = load_training_data(DATA_PATH)
    features, target = prepare_training_data(dataframe)

    evaluate_model_splits(dataframe, features, target)

    final_model = train_final_model(features, target)
    save_model(final_model, str(MODEL_PATH))


if __name__ == "__main__":
    main()
